# ClassLens ASD: Multimodal IEP Intelligence for Autistic Learners

**Gemma 4 Good Hackathon — Kaggle Submission**

ClassLens ASD is a multi-agent AI system that helps special education teachers convert daily classroom work artifacts into IEP-aligned progress data and personalized intervention materials.

### The Problem
- Special ed teachers spend **45-60 minutes per student per week** on manual IEP documentation
- Teachers with 8-12 students lose an entire workday weekly to paperwork
- Existing IEP software handles storage, not intelligence

### The Solution
Four Gemma 4 agents working in sequence:

| Agent | Gemma 4 Feature | What It Does |
|-------|----------------|-------------|
| 🔍 Vision Reader | Multimodal (image+text) | OCR: photo of student work → structured JSON |
| 🎯 IEP Mapper | Function Calling | Maps transcription to IEP goals, records trial data |
| 📊 Progress Analyst | Thinking Mode | Trend detection, regression alerts, explainable reasoning |
| ✏️ Material Forge | Function Calling | Generates 7 output types for teachers, parents, admin |

### Demo Students
Three fictional students with realistic ASD profiles created by Sarah Allan (15+ year special ed teacher):

| Student | Grade | ASD Level | Focus Area |
|---------|-------|-----------|------------|
| Maya | 3 | Level 2 | Communication, self-regulation |
| Jaylen | 1 | Level 3 (non-verbal) | Independence, visual schedules |
| Sofia | 5 | Level 1 | Executive function, writing |

## Setup

Install dependencies and configure the Gemma 4 API.

In [ ]:
# Install dependencies
!pip install -q google-genai pydantic python-dotenv Pillow plotly

In [ ]:
import os
import json
import sys
from pathlib import Path
from IPython.display import display, Markdown, Image

# If running from the notebooks/ directory, add parent to path
project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
    os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python {sys.version}")

In [ ]:
# Configure API key
# Option 1: Kaggle Secrets (recommended)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["GOOGLE_AI_STUDIO_KEY"] = secrets.get_secret("GOOGLE_AI_STUDIO_KEY")
    print("API key loaded from Kaggle Secrets")
except:
    pass

# Option 2: Set directly (for local testing)
# os.environ["GOOGLE_AI_STUDIO_KEY"] = "your-key-here"

# Check if API key is available
api_key = os.getenv("GOOGLE_AI_STUDIO_KEY", "")
USE_REAL_API = bool(api_key) and api_key != "your_api_key_here"

if USE_REAL_API:
    from core.gemma_client import GemmaClient
    client = GemmaClient()
    print(f"Using Gemma 4 API (model: {client.model})")
else:
    from tests.mock_api_responses import MockGemmaClient
    client = MockGemmaClient()
    print("No API key — using MockGemmaClient with realistic demo data")
    print("Set GOOGLE_AI_STUDIO_KEY in Kaggle Secrets for live inference")

## 1. Load Student Profiles

Each student has a JSON profile with demographics, IEP goals, sensory profile, interests, and trial history. These were designed by Sarah Allan based on real classroom patterns.

In [ ]:
# Load all student profiles
students = {}
for json_file in sorted(Path("data/students").glob("*.json")):
    with open(json_file) as f:
        data = json.load(f)
    students[data["student_id"]] = data

for sid, s in students.items():
    goals = ", ".join(g["domain"] for g in s["iep_goals"])
    interests = ", ".join(s["interests"][:2])
    print(f"{s['name']} (Grade {s['grade']}, ASD Level {s['asd_level']})")
    print(f"  Communication: {s['communication_level']}")
    print(f"  Interests: {interests}")
    print(f"  IEP Goals: {goals}")
    print(f"  Trial history: {sum(len(g.get('trial_history',[])) for g in s['iep_goals'])} entries")
    print()

In [ ]:
# Explore Maya's profile in detail
maya = students["maya_2026"]
display(Markdown(f"### Maya's Profile"))
display(Markdown(f"**Interests:** {', '.join(maya['interests'])}"))
display(Markdown(f"**Reinforcers:** {', '.join(maya['reinforcers'])}"))
display(Markdown(f"**Sensory seeks:** {', '.join(maya['sensory_profile']['seeks'])}"))
display(Markdown(f"**Sensory avoids:** {', '.join(maya['sensory_profile']['avoids'])}"))
display(Markdown(f"**Calming strategies:** {', '.join(maya['sensory_profile']['calming_strategies'])}"))

print("\nIEP Goals:")
for g in maya["iep_goals"]:
    baseline = g['baseline']['value'] if isinstance(g['baseline'], dict) else g['baseline']
    latest = g['trial_history'][-1]['pct'] if g.get('trial_history') else 'N/A'
    print(f"  [{g['goal_id']}] {g['domain']}: {g['description'][:80]}...")
    print(f"       Baseline: {baseline}% → Latest: {latest}% (Target: {g['target']}%)")

## 2. Agent 1: Vision Reader (Multimodal OCR)

The Vision Reader uses **Gemma 4's multimodal capability** to read a photo of student work and produce structured JSON. It uses **function calling** to output a schema-conforming transcription.

This replaces the teacher's manual data entry — snap a photo, get structured data.

In [ ]:
# Show the sample work image
image_path = "data/sample_work/maya_math_worksheet.png"
display(Markdown("### Maya's Math Worksheet"))
display(Image(filename=image_path, width=500))

In [ ]:
from agents.vision_reader import VisionReader

vision = VisionReader(client)

transcription = vision.transcribe(
    image_path=image_path,
    student_name="Maya",
    grade=3,
    asd_level=2,
    work_type="worksheet",
    task_description="Math counting and addition worksheet",
)

display(Markdown("### Structured Transcription (via Gemma 4 function calling)"))
print(json.dumps(transcription, indent=2))

In [ ]:
# Show the Gemma 4 function calling schema used
from schemas.tools import TRANSCRIBE_STUDENT_WORK

display(Markdown("### Function Calling Schema: `transcribe_student_work`"))
display(Markdown("This JSON schema is passed to `generate_with_tools()` — Gemma 4 returns structured output matching this schema."))
print(json.dumps(TRANSCRIBE_STUDENT_WORK, indent=2))

## 3. Agent 2: IEP Mapper (Function Calling)

The IEP Mapper takes the Vision Reader's transcription and **maps it to the student's IEP goals**. It uses Gemma 4 function calling to return structured goal matches with trial data.

This is the core intelligence — connecting raw classroom data to formal IEP tracking.

In [ ]:
from agents.iep_mapper import IEPMapper

mapper = IEPMapper(client, data_dir="data")

mapping = mapper.map_to_goals(
    student_id="maya_2026",
    transcription=transcription,
    work_type="worksheet",
)

display(Markdown("### Goal Mapping Result"))
for match in mapping.get("matched_goals", []):
    goal_id = match["goal_id"]
    goal = next((g for g in maya["iep_goals"] if g["goal_id"] == goal_id), {})
    print(f"Goal {goal_id} ({goal.get('domain', '?')})")
    print(f"  Trials: {match.get('trials', '?')} | Successes: {match.get('successes', '?')} | Pct: {match.get('percentage', '?')}%")
    print(f"  Reasoning: {match.get('reasoning', 'N/A')}")
    print()

In [ ]:
# Show the function calling schema
from schemas.tools import MAP_WORK_TO_GOALS

display(Markdown("### Function Calling Schema: `map_work_to_goals`"))
print(json.dumps(MAP_WORK_TO_GOALS, indent=2))

## 4. Agent 3: Progress Analyst (Thinking Mode)

The Progress Analyst uses **Gemma 4's thinking mode** (`thinking_budget_tokens=2048`) to perform explainable trend analysis. The thinking chain is preserved so teachers can understand *why* the AI reached its conclusions.

This is critical for trust — special ed teachers need to see the reasoning, not just a number.

In [ ]:
from agents.progress_analyst import ProgressAnalyst

analyst = ProgressAnalyst(client, data_dir="data")

# Analyze Maya's communication goal (G1) — the one with the strongest trend
analysis = analyst.analyze(student_id="maya_2026", goal_id="G1")

display(Markdown("### Progress Analysis: Maya Goal G1 (Communication)"))
display(Markdown(f"**Trend:** {analysis.get('trend', 'N/A')}"))
if analysis.get('alert'):
    display(Markdown(f"⚠️ **Alert:** {analysis.get('alert_message', '')}"))

display(Markdown("### Thinking Chain (Gemma 4 Thinking Mode)"))
display(Markdown("_This is the model's step-by-step reasoning — visible to teachers for trust and transparency._"))
print(analysis.get("thinking", "No thinking trace available"))

In [ ]:
display(Markdown("### Progress Note (Final Output)"))
print(analysis.get("progress_note", "No progress note generated"))

## 5. Visualize Progress Trends

Plotly charts show trial data over time for each IEP goal. Teachers and parents can see at a glance whether a student is improving, plateauing, or regressing.

In [ ]:
import plotly.graph_objects as go

# Plot Maya's goal progress across all 3 goals
fig = go.Figure()

colors = {"G1": "#5B8FB9", "G2": "#8BC34A", "G3": "#FF9800"}

for goal in maya["iep_goals"]:
    history = goal.get("trial_history", [])
    if not history:
        continue
    dates = [h["date"] for h in history]
    pcts = [h["pct"] for h in history]
    
    fig.add_trace(go.Scatter(
        x=dates, y=pcts,
        mode="lines+markers",
        name=f"{goal['goal_id']}: {goal['domain']}",
        line=dict(color=colors.get(goal['goal_id'], '#999'), width=3),
        marker=dict(size=10),
    ))
    
    # Add target line
    fig.add_hline(
        y=goal["target"],
        line_dash="dot",
        line_color=colors.get(goal['goal_id'], '#999'),
        annotation_text=f"{goal['goal_id']} target",
        opacity=0.5,
    )

fig.update_layout(
    title="Maya's IEP Goal Progress Over Time",
    xaxis_title="Date",
    yaxis_title="Success Rate (%)",
    yaxis=dict(range=[0, 105]),
    template="plotly_white",
    height=500,
    font=dict(size=14),
)

fig.show()

## 6. Agent 4: Material Forge (Function Calling)

The Material Forge generates **7 output types** for 3 audiences using Gemma 4 function calling:

| Output | Audience | Purpose |
|--------|----------|--------|
| Lesson Plan | Teacher | IEP-aligned lesson with student interests |
| Tracking Sheet | Teacher | Clipboard-ready data collection form |
| Social Story | Teacher | Carol Gray framework narrative |
| Visual Schedule | Teacher | Step-by-step routine with icons |
| First-Then Board | Teacher | Motivation board using reinforcers |
| Parent Communication | Parents | Warm, jargon-free progress update |
| Admin Report | Administration | Formal progress report for IEP meetings |

In [ ]:
from agents.material_forge import MaterialForge

forge = MaterialForge(client, data_dir="data")

# Generate a lesson plan for Maya's communication goal
lesson = forge.generate_lesson_plan(student_id="maya_2026", goal_id="G1")

display(Markdown("### Generated Lesson Plan"))
display(Markdown(f"**Title:** {lesson.get('title', lesson.get('lesson_title', 'N/A'))}"))
display(Markdown(f"**Duration:** {lesson.get('duration', lesson.get('estimated_duration_minutes', 'N/A'))}"))
display(Markdown(f"**Goal:** {lesson.get('objective', 'N/A')}"))

if lesson.get("materials"):
    display(Markdown("**Materials:** " + ", ".join(lesson["materials"])))
if lesson.get("activities"):
    display(Markdown("**Activities:**"))
    for act in lesson["activities"]:
        if isinstance(act, dict):
            print(f"  Step {act.get('step', '?')}: {act.get('description', act)}")
        else:
            print(f"  - {act}")
if lesson.get("sensory_supports"):
    display(Markdown("**Sensory Supports:** " + ", ".join(lesson["sensory_supports"])))

In [ ]:
# Generate a social story
story = forge.generate_social_story(
    student_id="maya_2026",
    scenario="greeting peers at school arrival",
    skill="responding to peer greetings",
)

display(Markdown("### Generated Social Story (Carol Gray Framework)"))
display(Markdown(f"**Title:** {story.get('title', 'N/A')}"))

pages = story.get("pages", [])
if pages:
    for i, page in enumerate(pages, 1):
        print(f"  Page {i}: {page}")
else:
    print(json.dumps(story, indent=2))

## 7. Full Pipeline: End-to-End

The pipeline chains all four agents: **Image → Transcription → Goal Mapping → Analysis**.

Material generation is separate (invoked from the UI) because teachers choose what to generate.

In [ ]:
from core.pipeline import ClassLensPipeline

pipeline = ClassLensPipeline(client=client, data_dir="data")

# Process Maya's behavior tally sheet
display(Markdown("### Processing: Maya's Peer Interaction Tally"))
display(Image(filename="data/sample_work/maya_behavior_tally.png", width=500))

In [ ]:
result = pipeline.process_work_artifact(
    student_id="maya_2026",
    image_path="data/sample_work/maya_behavior_tally.png",
    work_type="tally_sheet",
    subject="communication",
    date="2026-04-03",
)

display(Markdown("### Pipeline Result Summary"))
n_goals = len(result.get('goal_mapping', {}).get('matched_goals', []))
n_analyses = len(result.get('analyses', []))
n_alerts = len(result.get('alerts', []))
print(f"Goals matched: {n_goals}")
print(f"Analyses generated: {n_analyses}")
print(f"Alerts: {n_alerts}")

display(Markdown("### Transcription"))
print(json.dumps(result["transcription"], indent=2, default=str)[:1000])

In [ ]:
display(Markdown("### Goal Mapping"))
for match in result.get("goal_mapping", {}).get("matched_goals", []):
    print(f"  {match['goal_id']}: {match.get('percentage', '?')}% — {match.get('reasoning', '')}")

display(Markdown("### Analyses"))
for a in result.get("analyses", []):
    print(f"  {a.get('goal_id', '?')}: trend={a.get('trend', '?')}")
    if a.get("thinking"):
        print(f"    Thinking: {a['thinking'][:200]}...")

## 8. All Three Students

Run the pipeline for Jaylen and Sofia too — showing the system handles diverse ASD profiles.

In [ ]:
# Jaylen: non-verbal, Level 3, task checklist
display(Markdown("### Jaylen's Visual Schedule Checklist"))
display(Image(filename="data/sample_work/jaylen_task_checklist.png", width=500))

jaylen_result = pipeline.process_work_artifact(
    student_id="jaylen_2026",
    image_path="data/sample_work/jaylen_task_checklist.png",
    work_type="checklist",
    subject="daily_living",
    date="2026-04-03",
)

print(f"Goals matched: {len(jaylen_result.get('goal_mapping', {}).get('matched_goals', []))}")
for a in jaylen_result.get("analyses", []):
    print(f"  {a.get('goal_id', '?')}: trend={a.get('trend', '?')}")

In [ ]:
# Sofia: Level 1, gifted writer, free response
display(Markdown("### Sofia's Reflective Writing Sample"))
display(Image(filename="data/sample_work/sofia_writing_sample.png", width=500))

sofia_result = pipeline.process_work_artifact(
    student_id="sofia_2026",
    image_path="data/sample_work/sofia_writing_sample.png",
    work_type="free_response",
    subject="writing",
    date="2026-04-03",
)

print(f"Goals matched: {len(sofia_result.get('goal_mapping', {}).get('matched_goals', []))}")
for a in sofia_result.get("analyses", []):
    print(f"  {a.get('goal_id', '?')}: trend={a.get('trend', '?')}")

## 9. Architecture Deep Dive

### How We Use Gemma 4's Three Key Features

**1. Multimodal (Vision Reader)**
```python
# core/gemma_client.py — generate_multimodal()
contents = [
    types.Part.from_bytes(data=image_bytes, mime_type=mime_type),
    types.Part.from_text(text=prompt),
]
response = client.models.generate_content(model=model, contents=contents)
```

**2. Function Calling (Vision Reader, IEP Mapper, Material Forge)**
```python
# core/gemma_client.py — generate_with_tools()
config = types.GenerateContentConfig(
    tools=tools,
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(mode="AUTO")
    ),
)
# Returns: {"function": name, "args": {...}}
```

**3. Thinking Mode (Progress Analyst)**
```python
# core/gemma_client.py — generate_with_thinking()
config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget_tokens=2048),
)
# Returns: {"thinking": reasoning_chain, "output": final_response}
```

### Agent Pipeline Flow
```
Photo of student work
       │
       ▼
┌──────────────┐
│ Vision Reader │  ← Gemma 4 multimodal + function calling
│ (Agent 1)     │  → Structured transcription JSON
└──────┬───────┘
       │
       ▼
┌──────────────┐
│ IEP Mapper   │  ← Gemma 4 function calling
│ (Agent 2)    │  → Goal matches + trial data recorded
└──────┬───────┘
       │
       ▼
┌───────────────┐
│ Progress      │  ← Gemma 4 thinking mode
│ Analyst (3)   │  → Trend analysis + explainable reasoning
└──────┬────────┘
       │
       ▼
┌───────────────┐
│ Material      │  ← Gemma 4 function calling
│ Forge (4)     │  → 7 output types × 3 audiences
└───────────────┘
```

## 10. Impact & What's Next

### Time Savings
| Task | Before ClassLens | After ClassLens |
|------|-----------------|----------------|
| Read & transcribe student work | 10-15 min/artifact | 30 seconds |
| Map to IEP goals & record trials | 15-20 min | Automatic |
| Write progress notes | 10-15 min/goal | 1 click |
| Create lesson plans | 30-45 min | 1 click |
| Draft parent communications | 15-20 min | 1 click + review |
| **Total per student per week** | **45-60 min** | **~5 min** |

### Design Principles
- **Teacher-in-the-loop:** Every output has approve/edit/regenerate. Nothing auto-sends.
- **ASD-friendly:** Predictable layouts, calm colors, concrete vocabulary
- **Privacy-first:** FERPA-aware design, no student PII in API calls, local data storage
- **Pre-baked demo:** All sample results cached — demo never waits for API

### Links
- **GitHub:** [jallanUSF/classlens-asd](https://github.com/jallanUSF/classlens-asd)
- **Live Demo:** See Streamlit app deployment
- **Video:** See competition submission

---

*Built by Jeff & Sarah Allan for the Gemma 4 Good Hackathon 2026.*

*ClassLens ASD: Reclaiming teacher time, transforming student outcomes.*